# Radar de vagas — caderno Python

Guia reproduzível da análise descritiva usada no dashboard, preparado com assistência de IA. Não é um registro retroativo de pesquisa nem uma afirmação de autoria independente. Leia, execute e adapte às regras de atribuição da sua instituição.

**Pergunta:** como explorar os rótulos históricos sem transformar associações em um detector de fraude?

Instale `requirements.txt`, selecione o ambiente Python 3.12/3.13 no Jupyter e execute as células em ordem a partir da raiz do projeto. O notebook usa somente a base local, sem credenciais.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import plotly.graph_objects as go

root = Path.cwd()
assert (root / "data/jobs_database.db").exists(), "Execute na raiz do projeto"
with sqlite3.connect(f"file:{root / 'data/jobs_database.db'}?mode=ro", uri=True) as conn:
    df = pd.read_sql_query("SELECT * FROM vagas ORDER BY id_vaga", conn)
assert df.id_vaga.is_unique
assert set(df.fraude_bin.unique()) <= {0, 1}
print(f"{len(df):,} anúncios; {len(df.columns)} colunas; IDs únicos")


10,000 anúncios; 31 colunas; IDs únicos


## 1. Denominador antes da cor

`fraude_bin` é o rótulo existente, não uma previsão. A taxa descritiva é `soma dos rótulos / quantidade de anúncios`. A base canônica foi preservada do projeto original; o repositório não documenta aqui uma coleta nova ou validação externa dos rótulos.


In [2]:
total = len(df)
fraudes = int(df.fraude_bin.sum())
taxa = fraudes / total if total else 0
print(f"Fraudulentas: {fraudes}; reais: {total - fraudes}; taxa: {taxa:.2%}")
print(df.groupby("status_vaga").size().to_string())


Fraudulentas: 453; reais: 9547; taxa: 4.53%
status_vaga
Fraudulenta     453
Real           9547


## 2. Recortes combinados

Compare uma parte com a base, mas mantenha os denominadores explícitos. Filtrar pelo próprio rótulo torna uma taxa de 100% trivial: não mede risco futuro.


In [3]:
recorte = df[(df.tem_logo == "Não") & (df.remoto == "Sim")]
n = len(recorte)
f = int(recorte.fraude_bin.sum())
print(f"Sem logo E remoto: n={n}, rótulos de fraude={f}, taxa={f/n:.2%}" if n else "Sem anúncios")
print("Diferença para a base (pontos percentuais):", round(100 * (f/n - taxa), 2) if n else None)


Sem logo E remoto: n=120, rótulos de fraude=26, taxa=21.67%
Diferença para a base (pontos percentuais): 17.14


## 3. Fluxos: contagem, não causalidade

O Sankey da web combina duas tabelas de contingência. Cada transição soma a mesma população. A etapa intermediária agrega registros; não permite inferir um caminho individual logo → remoto → rótulo, nem uma explicação causal.


In [4]:
primeira_etapa = df.groupby(["tem_logo", "remoto"]).size()
segunda_etapa = df.groupby(["remoto", "status_vaga"]).size()
assert primeira_etapa.sum() == segunda_etapa.sum() == total
print("Logo → remoto")
print(primeira_etapa.to_string())
print("\nRemoto → rótulo")
print(segunda_etapa.to_string())


Logo → remoto
tem_logo  remoto
Não       Não       1960
          Sim        120
Sim       Não       7586
          Sim        334

Remoto → rótulo
remoto  status_vaga
Não     Fraudulenta     413
        Real           9133
Sim     Fraudulenta      40
        Real            414


## 4. Distribuições de texto

A distribuição acumulada empírica (ECDF) responde: qual proporção tem descrição com **até** x caracteres? Cada grupo tem seu denominador; a curva chega a 100%. A mediana é o ponto de 50%. Tamanho não é qualidade, idioma, credibilidade ou prova de fraude.


In [5]:
fig = go.Figure()
for label, color in [("Real", "#65734f"), ("Fraudulenta", "#ad4035")]:
    valores = df.loc[df.status_vaga == label, "tamanho_descricao"]
    frequencias = valores.value_counts().sort_index()
    acumulada = 100 * frequencias.cumsum() / len(valores)
    assert acumulada.iloc[-1] == 100
    fig.add_trace(go.Scatter(x=[0, *frequencias.index], y=[0, *acumulada],
                             mode="lines", name=label, line={"color": color, "shape": "hv"}))
    print(f"{label}: n={len(valores)}, mediana={valores.median():.0f} caracteres")
fig.update_layout(xaxis_title="Caracteres", yaxis_title="Anúncios com até esse tamanho (%)")
# Execute fig.show() no Jupyter para explorar o gráfico interativo.


Real: n=9547, mediana=993 caracteres
Fraudulenta: n=453, mediana=946 caracteres


## 5. Casos e limites

O caderno de anúncios da web mostra o menor ID de cada rótulo disponível no recorte. A regra é reproduzível, mas não produz uma amostra representativa ou casos equivalentes. Não selecione exemplos apenas por confirmarem uma hipótese.


In [6]:
ids = df.groupby("status_vaga").id_vaga.min()
print(df[df.id_vaga.isin(ids)][["id_vaga", "title", "status_vaga", "pais"]].to_string(index=False))


 id_vaga            title status_vaga pais
       1 Marketing Intern        Real   US
      99  IC&E Technician Fraudulenta   US


## 6. Reproduzir e explicar

- Web: `python app.py`. Os callbacks consultam SQLite com parâmetros, recalculam gráficos e retornam registros do recorte.
- Excel interativo: `python scripts/build_workbook.py`. As seis listas alimentam fórmulas SUMPRODUCT e gráficos nativos; não há macros.
- Excel somente dados: `Jobs_Data.xlsx`. Não contém gráficos ou fórmulas.
- Testes: `python -m pytest -q` após instalar `requirements-dev.txt`.

### Questões para defender a análise
1. Por que a taxa muda ao filtrar por setor? Qual denominador você está usando?
2. Por que duas curvas diferentes não demonstram causalidade?
3. O que se perde quando um Sankey agrega a etapa intermediária?
4. Quais informações de proveniência e amostragem ainda precisam ser documentadas a partir da fonte original?

Não há validação de modelo, previsão, teste de causalidade ou certificação de acessibilidade. Dados históricos e desequilíbrio de rótulos limitam a generalização.
